#Análise de Qualidade dos Dados

In [0]:
# Define as tabelas da camada Bronze que serão analisadas
tabelas = [
    "customers",
    "geolocation",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "products",
    "sellers",
    "category_translation"
]

# Define o catálogo e o schema de origem dos dados
catalogo = "workspace"
schema = "bronze"

In [0]:
# Levanta a quantidade de registros e colunas de cada tabela Bronze
resultados = []

for tabela in tabelas:

    df = spark.table(f"{catalogo}.{schema}.{tabela}")

    resultados.append({
        "tabela": tabela,
        "registros": df.count(),
        "colunas": len(df.columns)
    })

# Exibe o inventário para validar a carga inicial
display(spark.createDataFrame(resultados))

colunas,registros,tabela
7,99441,customers
7,1000163,geolocation
9,112650,order_items
7,103886,order_payments
9,104162,order_reviews
10,99441,orders
11,32951,products
6,3095,sellers
4,71,category_translation


In [0]:
from pyspark.sql import functions as F

# Verifica valores nulos e vazios em todas as colunas da camada Bronze
resultados_nulos = []

for tabela in tabelas:

    df = spark.table(f"{catalogo}.{schema}.{tabela}")
    total = df.count()

    for coluna in df.columns:

        # Conta registros com valores nulos ou strings vazias
        quantidade_nulos = df.filter(
            F.col(f"`{coluna}`").isNull() |
            (F.trim(F.col(f"`{coluna}`").cast("string")) == "")
        ).count()

        # Calcula o percentual de registros afetados
        percentual = (
            quantidade_nulos / total * 100
            if total > 0 else 0
        )

        resultados_nulos.append({
            "tabela": tabela,
            "coluna": coluna,
            "total_registros": total,
            "quantidade_nulos": quantidade_nulos,
            "percentual_nulos": round(percentual, 2)
        })

# Consolida os resultados da análise de completude
df_nulos = spark.createDataFrame(resultados_nulos)

# Exibe somente as colunas que apresentam valores nulos ou vazios
display(
    df_nulos
    .filter(F.col("quantidade_nulos") > 0)
    .orderBy(F.desc("percentual_nulos"))
)

coluna,percentual_nulos,quantidade_nulos,tabela,total_registros
review_comment_title,88.34,87658,order_reviews,99224
review_comment_message,58.71,58256,order_reviews,99224
order_delivered_customer_date,2.98,2965,orders,99441
product_category_name,1.85,610,products,32951
product_name_lenght,1.85,610,products,32951
product_description_lenght,1.85,610,products,32951
product_photos_qty,1.85,610,products,32951
order_delivered_carrier_date,1.79,1783,orders,99441
order_approved_at,0.16,160,orders,99441
product_weight_g,0.01,2,products,32951


In [0]:
from pyspark.sql import functions as F

# Define as chaves esperadas para cada tabela Bronze
chaves = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id", "order_id"],
    "category_translation": ["product_category_name"]
}

resultados_duplicatas = []

# Verifica se existem combinações de chaves repetidas
for tabela, colunas_chave in chaves.items():

    df = spark.table(f"{catalogo}.{schema}.{tabela}")

    total = df.count()

    # Identifica registros com chaves repetidas
    duplicatas = (
        df.groupBy(*colunas_chave)
        .count()
        .filter(F.col("count") > 1)
    )

    quantidade_duplicatas = duplicatas.agg(
        F.sum(F.col("count") - 1)
    ).collect()[0][0] or 0

    resultados_duplicatas.append({
        "tabela": tabela,
        "chave": ", ".join(colunas_chave),
        "total_registros": total,
        "registros_duplicados_excedentes": quantidade_duplicatas
    })

# Exibe o diagnóstico de unicidade das tabelas
display(spark.createDataFrame(resultados_duplicatas))

chave,registros_duplicados_excedentes,tabela,total_registros
customer_id,0,customers,99441
order_id,0,orders,99441
product_id,0,products,32951
seller_id,0,sellers,3095
"order_id, order_item_id",0,order_items,112650
"order_id, payment_sequential",0,order_payments,103886
"review_id, order_id",0,order_reviews,99224
product_category_name,0,category_translation,71


In [0]:
from pyspark.sql import functions as F

# Carrega a tabela de avaliações da camada Bronze
df_reviews = spark.table("workspace.bronze.order_reviews")

# Identifica combinações de review_id e order_id repetidas
reviews_duplicadas = (
    df_reviews
    .groupBy("review_id", "order_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

# Exibe as chaves duplicadas e suas frequências
display(reviews_duplicadas)

review_id,order_id,count


In [0]:
# Seleciona apenas as colunas originais, ignorando metadados de ingestão
colunas_originais = [
    coluna for coluna in df_reviews.columns
    if not coluna.startswith("_")
]

# Conta registros duplicados considerando todas as colunas originais
total = df_reviews.count()

registros_unicos = (
    df_reviews
    .select(*colunas_originais)
    .distinct()
    .count()
)

duplicatas_completas = total - registros_unicos

# Exibe o resultado da verificação
print(f"Total de registros: {total}")
print(f"Registros distintos: {registros_unicos}")
print(f"Duplicatas completas excedentes: {duplicatas_completas}")

Total de registros: 99224
Registros distintos: 99224
Duplicatas completas excedentes: 0


In [0]:
from pyspark.sql import functions as F

# Define os relacionamentos esperados entre as tabelas Bronze
relacionamentos = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("order_items", "seller_id", "sellers", "seller_id"),
    ("order_payments", "order_id", "orders", "order_id"),
    ("order_reviews", "order_id", "orders", "order_id")
]

resultados_integridade = []

# Verifica registros que não possuem correspondência na tabela relacionada
for tabela_origem, chave_origem, tabela_destino, chave_destino in relacionamentos:

    df_origem = spark.table(f"{catalogo}.{schema}.{tabela_origem}")
    df_destino = spark.table(f"{catalogo}.{schema}.{tabela_destino}")

    # Identifica registros órfãos por meio de LEFT ANTI JOIN
    registros_orfaos = (
        df_origem.join(
            df_destino.select(chave_destino).distinct(),
            df_origem[chave_origem] == df_destino[chave_destino],
            "left_anti"
        )
        .count()
    )

    resultados_integridade.append({
        "tabela_origem": tabela_origem,
        "chave_origem": chave_origem,
        "tabela_destino": tabela_destino,
        "chave_destino": chave_destino,
        "registros_orfaos": registros_orfaos
    })

# Exibe o diagnóstico de integridade referencial
display(spark.createDataFrame(resultados_integridade))

chave_destino,chave_origem,registros_orfaos,tabela_destino,tabela_origem
customer_id,customer_id,0,customers,orders
order_id,order_id,0,orders,order_items
product_id,product_id,0,products,order_items
seller_id,seller_id,0,sellers,order_items
order_id,order_id,0,orders,order_payments
order_id,order_id,0,orders,order_reviews


In [0]:
from pyspark.sql import functions as F

# Carrega as tabelas financeiras da camada Bronze
df_items = spark.table("workspace.bronze.order_items")
df_payments = spark.table("workspace.bronze.order_payments")

# Verifica preços e fretes negativos ou inválidos
resultado_items = df_items.select(
    F.sum(
        F.when(F.col("price").cast("double") < 0, 1).otherwise(0)
    ).alias("precos_negativos"),

    F.sum(
        F.when(F.col("freight_value").cast("double") < 0, 1).otherwise(0)
    ).alias("fretes_negativos"),

    F.sum(
        F.when(
            F.col("price").isNotNull() &
            F.col("price").cast("double").isNull(), 1
        ).otherwise(0)
    ).alias("precos_invalidos")
)

# Verifica valores negativos nos pagamentos
resultado_payments = df_payments.select(
    F.sum(
        F.when(F.col("payment_value").cast("double") < 0, 1)
        .otherwise(0)
    ).alias("pagamentos_negativos")
)

display(resultado_items)
display(resultado_payments)

precos_negativos,fretes_negativos,precos_invalidos
0,0,0


pagamentos_negativos
0


In [0]:
# Carrega os pedidos da camada Bronze
df_orders = spark.table("workspace.bronze.orders")

# Converte temporariamente as datas para realizar as verificações
df_orders_qualidade = (
    df_orders
    .withColumn(
        "data_compra",
        F.to_timestamp("order_purchase_timestamp")
    )
    .withColumn(
        "data_entrega",
        F.to_timestamp("order_delivered_customer_date")
    )
)

# Verifica inconsistências entre compra, entrega e status
resultado_entregas = df_orders_qualidade.select(

    F.sum(
        F.when(
            F.col("data_entrega") < F.col("data_compra"), 1
        ).otherwise(0)
    ).alias("entregas_antes_da_compra"),

    F.sum(
        F.when(
            (F.col("order_status") == "delivered") &
            F.col("data_entrega").isNull(), 1
        ).otherwise(0)
    ).alias("entregues_sem_data"),

    F.sum(
        F.when(
            (F.col("order_status") != "delivered") &
            F.col("data_entrega").isNotNull(), 1
        ).otherwise(0)
    ).alias("nao_entregues_com_data")
)

display(resultado_entregas)

entregas_antes_da_compra,entregues_sem_data,nao_entregues_com_data
0,8,6


In [0]:
# Carrega as avaliações da camada Bronze
df_reviews = spark.table("workspace.bronze.order_reviews")

# Verifica notas fora do intervalo esperado de 1 a 5
resultado_avaliacoes = df_reviews.select(

    F.sum(
        F.when(
            ~F.col("review_score").cast("int").between(1, 5),
            1
        ).otherwise(0)
    ).alias("notas_fora_do_intervalo"),

    F.sum(
        F.when(
            F.col("review_score").isNull(), 1
        ).otherwise(0)
    ).alias("notas_nulas")
)

display(resultado_avaliacoes)

notas_fora_do_intervalo,notas_nulas
0,0


In [0]:
from pyspark.sql import functions as F

# Carrega os pedidos da camada Bronze
df_orders = spark.table("workspace.bronze.orders")

# Identifica pedidos com inconsistências entre status e data de entrega
pedidos_inconsistentes = (
    df_orders
    .filter(
        (
            (F.col("order_status") == "delivered") &
            F.col("order_delivered_customer_date").isNull()
        )
        |
        (
            (F.col("order_status") != "delivered") &
            F.col("order_delivered_customer_date").isNotNull()
        )
    )
    .select(
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    )
)

# Exibe os registros para investigar as inconsistências
display(pedidos_inconsistentes)

order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date
1950d777989f6a877539f53795b4c3c3,canceled,2018-02-19 19:48:52,2018-03-21 22:03:51,2018-03-09 00:00:00
2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,null,2017-12-18 00:00:00
dabf2b0e35b423f94618bf965fcb7514,canceled,2016-10-09 00:56:52,2016-10-16 14:36:59,2016-11-30 00:00:00
f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,null,2018-07-16 00:00:00
2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,null,2018-07-30 00:00:00
770d331c84e5b214bd9dc70a10b829d0,canceled,2016-10-07 14:52:30,2016-10-14 15:07:11,2016-11-29 00:00:00
8beb59392e21af5eb9547ae1a9938d06,canceled,2016-10-08 20:17:50,2016-10-19 18:47:43,2016-11-30 00:00:00
e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,null,2018-07-30 00:00:00
0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,null,2018-07-24 00:00:00
65d1e226dfaeb8cdc42f665422522d14,canceled,2016-10-03 21:01:41,2016-11-08 10:58:34,2016-11-25 00:00:00


In [0]:
from pyspark.sql import functions as F

# Carrega os itens dos pedidos
df_items = spark.table("workspace.bronze.order_items")

# Converte temporariamente os valores para análise estatística
df_valores = (
    df_items
    .withColumn("preco", F.col("price").cast("double"))
    .withColumn("frete", F.col("freight_value").cast("double"))
)

# Calcula estatísticas descritivas de preços e fretes
display(
    df_valores.select(
        "preco",
        "frete"
    ).summary(
        "count",
        "min",
        "25%",
        "50%",
        "75%",
        "max",
        "mean",
        "stddev"
    )
)

summary,preco,frete
count,112650,112650
min,0.85,0.0
25%,39.9,13.08
50%,74.99,16.26
75%,134.9,21.15
max,6735.0,409.68
mean,120.65373901477311,19.99031992898562
stddev,183.63392805025924,15.806405412297073


In [0]:
from pyspark.sql import functions as F

# Calcula os quartis de preço e frete
quartis_preco = df_valores.approxQuantile(
    "preco", [0.25, 0.75], 0.01
)

quartis_frete = df_valores.approxQuantile(
    "frete", [0.25, 0.75], 0.01
)

# Calcula os limites superiores pelo método IQR
iqr_preco = quartis_preco[1] - quartis_preco[0]
iqr_frete = quartis_frete[1] - quartis_frete[0]

limite_preco = quartis_preco[1] + 1.5 * iqr_preco
limite_frete = quartis_frete[1] + 1.5 * iqr_frete

# Conta os registros acima dos limites calculados
outliers_preco = df_valores.filter(
    F.col("preco") > limite_preco
).count()

outliers_frete = df_valores.filter(
    F.col("frete") > limite_frete
).count()

# Exibe os resultados da análise
print(f"Limite superior de preço: R$ {limite_preco:.2f}")
print(f"Quantidade de outliers de preço: {outliers_preco}")

print(f"Limite superior de frete: R$ {limite_frete:.2f}")
print(f"Quantidade de outliers de frete: {outliers_frete}")

Limite superior de preço: R$ 265.12
Quantidade de outliers de preço: 8896
Limite superior de frete: R$ 32.83
Quantidade de outliers de frete: 11815
